# LLM Baselines Pipeline

This notebook orchestrates the LLM baselines evaluation pipeline:
1. Precompute gold source embeddings for semantic matching
2. Test mode: Run 5 records on all models to verify pipeline
3. Full run: Launch parallel model executions
4. Aggregate all results and compute metrics
5. Visualize comparison across models

In [1]:
# =============================================================================
# CELL 1: Setup and Imports
# =============================================================================
import sys
import os
import subprocess
import pandas as pd
import numpy as np
import json
import glob
from typing import List, Dict
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams
import warnings
warnings.filterwarnings('ignore')
from dotenv import load_dotenv
import os

load_dotenv()  # loads .env into os.environ


# Import local modules
from config import (
    MODEL_LIST, 
    RESULTS_DIR, 
    SCAR_PATH,
    TEST_MODE_RECORD_LIMIT,
    NUM_ANALOGIES,
    get_output_path,
    get_output_filename
)

print(f"Available Models ({len(MODEL_LIST)}):")
for i, m in enumerate(MODEL_LIST, 1):
    print(f"  {i:2d}. {m}")

print(f"\nResults directory: {RESULTS_DIR}")
print(f"SCAR dataset: {SCAR_PATH}")
print(f"Number of analogies per target: {NUM_ANALOGIES}")

Available Models (12):
   1. gpt-oss-20b
   2. gpt-oss-120b
   3. gpt-4.1-mini
   4. gpt-4.1-nano
   5. grok-4-fast
   6. gemini-2.5-flash-lite
   7. llama-3.1-405b-instruct
   8. meta-llama-3-1-70b-instruct
   9. meta-llama-3-1-8b-instruct
  10. deepseek-r1
  11. qwen3-14b
  12. qwen3-32b

Results directory: d:\My_working_area\Masters\Thesis\code\Toward-Usable-Scientific-Analogies\Toward-Usable-Scientific-Analogies\stage_2_Modular_solution\LLM\results
SCAR dataset: d:\My_working_area\Masters\Thesis\code\Toward-Usable-Scientific-Analogies\Toward-Usable-Scientific-Analogies\stage_2_Modular_solution\LLM\..\..\data\SCAR_cleaned_manually.csv
Number of analogies per target: 20


In [2]:
# =============================================================================
# CELL 2: Precompute Embeddings (Run Once)
# =============================================================================
from precompute_similarity import (
    precompute_gold_embeddings, 
    precompute_target_embeddings,
    SemanticMatcher,
    test_semantic_matcher
)

# Set to True to force recomputation even if pkl files exist
# Set to False to skip precomputation if pkl files already exist
FORCE_RECOMPUTE = False

# Step 1: Precompute gold source embeddings (for semantic matching)
# NOTE: Unified evaluation for both targetonly and withsub modes
# Compares ONLY source concept names (no sub-concepts)
print("=" * 70)
print("STEP 1: Gold source embeddings (for semantic matching)")
print("NOTE: Unified evaluation for both targetonly and withsub modes")
print("=" * 70)
precompute_gold_embeddings(force=FORCE_RECOMPUTE)

# Step 2: Precompute target embeddings (for top-1-embedding selection)
# Creates two files:
#   - target_embeddings.pkl (target-only, for targetonly mode)
#   - target_with_subconcepts_embeddings.pkl (target+subconcepts, for withsub mode)
print("\n" + "=" * 70)
print("STEP 2: Target embeddings (for top-1-embedding selection)")
print("Creates two files: target-only and target-with-subconcepts")
print("Uses OpenAI text-embedding-3-small for top-1-embedding selection")
print("=" * 70)
precompute_target_embeddings(force=FORCE_RECOMPUTE)

# Step 3: Test the semantic matcher
print("\n" + "=" * 70)
print("STEP 3: Testing Semantic Matcher")
print("=" * 70)
test_semantic_matcher()

STEP 1: Gold source embeddings (for semantic matching)
NOTE: Unified evaluation for both targetonly and withsub modes
Gold Source Embeddings Already Exist
File: d:\My_working_area\Masters\Thesis\code\Toward-Usable-Scientific-Analogies\Toward-Usable-Scientific-Analogies\stage_2_Modular_solution\LLM\gold_source_embeddings.pkl
Skipping precomputation. Use force=True to recompute.

STEP 2: Target embeddings (for top-1-embedding selection)
Creates two files: target-only and target-with-subconcepts
Uses OpenAI text-embedding-3-small for top-1-embedding selection
Target Embeddings Already Exist
File 1: d:\My_working_area\Masters\Thesis\code\Toward-Usable-Scientific-Analogies\Toward-Usable-Scientific-Analogies\stage_2_Modular_solution\LLM\target_embeddings.pkl
File 2: d:\My_working_area\Masters\Thesis\code\Toward-Usable-Scientific-Analogies\Toward-Usable-Scientific-Analogies\stage_2_Modular_solution\LLM\target_with_subconcepts_embeddings.pkl
Skipping precomputation. Use force=True to recompute

In [3]:
# =============================================================================
# CELL 3: Test Mode - Run 5 Records on All Models
# =============================================================================
from run_model import run_model
from evaluate_model import evaluate_model_results
import dspy

def run_test_mode(models_to_test: List[str] = None, modes: List[str] = None):
    """
    Run test mode on specified models and modes.
    
    Args:
        models_to_test: List of model names (default: first 2 models)
        modes: List of modes (default: both)
    """
    if models_to_test is None:
        models_to_test = MODEL_LIST[:2]  # Test first 2 models
    if modes is None:
        #modes = ["targetonly"]
        modes = ["targetonly", "withsub"]
    
    print("="*70)
    print("TEST MODE: Running pipeline validation")
    print(f"Models: {models_to_test}")
    print(f"Modes: {modes}")
    print(f"Records per run: {TEST_MODE_RECORD_LIMIT}")
    print("="*70)
    
    test_results = []
    
    for model in models_to_test:
        for mode in modes:
            print(f"\n{'='*70}")
            print(f"Testing: {model} | Mode: {mode}")
            print("="*70)
            
            try:
                # Step 1: Generate analogies
                print("\n--- GENERATION ---")
                gen_df = run_model(
                    model_name=model,
                    mode=mode,
                    test_mode=True,
                    verbose=True
                )
                
                # Step 2: Evaluate
                print("\n--- EVALUATION ---")
                input_path = get_output_path(model, mode, is_eval=False)
                eval_df = evaluate_model_results(
                    input_path=input_path,
                    test_mode=True,
                    verbose=True
                )
                
                test_results.append({
                    'model': model,
                    'mode': mode,
                    'status': 'success',
                    'records': len(gen_df),
                    'avg_score': eval_df['average_score'].mean()
                })
                
            except Exception as e:
                print(f"ERROR: {e}")
                test_results.append({
                    'model': model,
                    'mode': mode,
                    'status': f'error: {str(e)}',
                    'records': 0,
                    'avg_score': None
                })
    
    # Summary
    print("\n" + "="*70)
    print("TEST MODE SUMMARY")
    print("="*70)
    summary_df = pd.DataFrame(test_results)
    print(summary_df.to_string(index=False))
    
    return summary_df

# Run test mode on first 2 models (uncomment to run)
test_summary = run_test_mode(models_to_test=MODEL_LIST[:2])

TEST MODE: Running pipeline validation
Models: ['gpt-oss-20b', 'gpt-oss-120b']
Modes: ['targetonly', 'withsub']
Records per run: 3

Testing: gpt-oss-20b | Mode: targetonly

--- GENERATION ---
Running Model: gpt-oss-20b
Mode: targetonly
Test Mode: True

Loading SCAR dataset from d:\My_working_area\Masters\Thesis\code\Toward-Usable-Scientific-Analogies\Toward-Usable-Scientific-Analogies\stage_2_Modular_solution\LLM\..\..\data\SCAR_cleaned_manually.csv
Loaded 400 records
Deduplicated 400 records to 321 unique targets
Test mode: Using first 3 unique targets

Initializing LLM client for gpt-oss-20b...

Generating analogies for 3 records...


Processing gpt-oss-20b:   0%|          | 0/3 [00:00<?, ?it/s]


Record ID: 1
Target: biological clock
Gold Sources (1): ['clock']
Mode: targetonly
Sub-concepts passed to generator: ''


Processing gpt-oss-20b:  33%|███▎      | 1/3 [00:04<00:08,  4.41s/it]

Generated Analogies:
  1. Alarm clock
  2. Traffic light
  3. Metronome
  4. Hourglass
  5. Calendar
  6. Sun dial
  7. Moon phase
  8. Heartbeat
  9. Pulse monitor
  10. Seasons
  11. Stopwatch
  12. Sine wave
  13. Clock tower
  14. Time baton
  15. Rain timer
  16. Weather vane
  17. Plant growth
  18. Ticking toy
  19. Sleep cycle
  20. Garden schedule

Record ID: 2
Target: Biosphere
Gold Sources (1): ['Library']
Mode: targetonly
Sub-concepts passed to generator: ''


...


LLM Judge Scores (for top1_baseline: 'alarm clock'):
  Coherence: 3
  Mapping: 3
  Explanatory: 3
  Average: 3.0

LLM Judge Scores (for top1_embedding: 'computer timer'):
  Coherence: 3
  Mapping: 2
  Explanatory: 2
  Average: 2.33

Gold Matching:
  Exact Ranks (all): [1]
  Semantic Ranks (all): [1]
  Found Generated Analogies (exact match): ['alarm clock']
  Found Generated Analogies (semantic match): []
  Exact Ranks by generated analogy: {'alarm clock': 1}
  Semantic Ranks by generated analogy: {'alarm clock': 1}

Similarity per Gold Source:
  clock: highest=0.7661, avg=0.2999

Record ID: 2
Target: Biosphere
Target Sub-concepts: biology, biodiversity, ecosystem
Gold Sources (1): ['Library']
Top-1 Baseline (model order): human body
Top-1 Embedding (by similarity): ecosystem services (score: 0.5383)
All Generated: ['human body', 'city', 'computer network', 'garden', 'orchestra', 'economy', 'library', 'school', 'solar system', 'ecosystem services', 'traffic system', 'forest canopy', '

Evaluating:  67%|██████▋   | 2/3 [00:17<00:09,  9.13s/it]


LLM Judge Scores (for top1_baseline: 'human body'):
  Coherence: 3
  Mapping: 3
  Explanatory: 3
  Average: 3.0

LLM Judge Scores (for top1_embedding: 'ecosystem services'):
  Coherence: 2
  Mapping: 2
  Explanatory: 2
  Average: 2.0

Gold Matching:
  Exact Ranks (all): [7]
  Semantic Ranks (all): [7]
  Found Generated Analogies (exact match): ['library']
  Found Generated Analogies (semantic match): []
  Exact Ranks by generated analogy: {'library': 7}
  Semantic Ranks by generated analogy: {'library': 7}

Similarity per Gold Source:
  Library: highest=1.0, avg=0.2829

Record ID: 3
Target: Respiratory system
Target Sub-concepts: oxygen, the lungs, breathing muscles
Gold Sources (3): ['engine', 'Air handling system', 'air circulation ducts']
Top-1 Baseline (model order): Air conditioner
Top-1 Embedding (by similarity): Yoga breathing (score: 0.5955)
All Generated: ['Air conditioner', 'Car engine', 'Bellows', 'Tree roots', 'Traffic system', 'Water filter', 'Pump', 'Ventilator', 'Siphon

Evaluating: 100%|██████████| 3/3 [00:23<00:00,  7.88s/it]


LLM Judge Scores (for top1_baseline: 'Air conditioner'):
  Coherence: 2
  Mapping: 2
  Explanatory: 2
  Average: 2.0

LLM Judge Scores (for top1_embedding: 'Yoga breathing'):
  Coherence: 2
  Mapping: 2
  Explanatory: 2
  Average: 2.0

Gold Matching:
  Exact Ranks (all): [2]
  Semantic Ranks (all): [1, 2]
  Found Generated Analogies (exact match): ['Car engine']
  Found Generated Analogies (semantic match): ['Air conditioner']
  Exact Ranks by generated analogy: {'Car engine': 2}
  Semantic Ranks by generated analogy: {'Car engine': 2, 'Air conditioner': 1}

Similarity per Gold Source:
  engine: highest=0.8576, avg=0.3048
  Air handling system: highest=0.6041, avg=0.2428
  air circulation ducts: highest=0.4696, avg=0.1856

EVALUATION SUMMARY
Total records: 3
Successful evaluations: 3
Errors: 0

LLM Judge Scores - TOP1_BASELINE (mean):
  Coherence: 2.67
  Mapping: 2.67
  Explanatory: 2.67
  Average: 2.67

LLM Judge Scores - TOP1_EMBEDDING (mean):
  Average: 2.11

Hit@K (Exact Match - a

In [ ]:
# =============================================================================
# CELL 4: Full Run Orchestration (Parallel Execution)
# =============================================================================

def run_single_model_subprocess(model: str, mode: str):
    """
    Run a single model as a subprocess for parallel execution.
    """
    cmd = [
        sys.executable,
        "run_model.py",
        "--model", model,
        "--mode", mode
    ]
    
    print(f"Starting: {model} | {mode}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode == 0:
        print(f"Completed: {model} | {mode}")
        return {'model': model, 'mode': mode, 'status': 'success'}
    else:
        print(f"Failed: {model} | {mode}\n{result.stderr[:500]}")
        return {'model': model, 'mode': mode, 'status': f'error: {result.stderr[:200]}'}


def evaluate_single_model_subprocess(model: str, mode: str):
    """
    Evaluate a single model's results as a subprocess.
    """
    input_path = get_output_path(model, mode, is_eval=False)
    
    cmd = [
        sys.executable,
        "evaluate_model.py",
        "--input", input_path
    ]
    
    print(f"Evaluating: {model} | {mode}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode == 0:
        print(f"Evaluation completed: {model} | {mode}")
        return {'model': model, 'mode': mode, 'status': 'success'}
    else:
        print(f"Evaluation failed: {model} | {mode}\n{result.stderr[:500]}")
        return {'model': model, 'mode': mode, 'status': f'error: {result.stderr[:200]}'}


def run_full_pipeline(models: List[str] = None, max_workers: int = 4):
    """
    Run the full pipeline for all models in parallel.
    
    Args:
        models: List of model names (default: all models)
        max_workers: Maximum parallel processes
    """
    if models is None:
        models = MODEL_LIST
    
    modes = ["targetonly", "withsub"]
    
    print("="*70)
    print("FULL PIPELINE EXECUTION")
    print(f"Models: {len(models)}")
    print(f"Modes: {modes}")
    print(f"Total runs: {len(models) * len(modes)}")
    print(f"Max workers: {max_workers}")
    print("="*70)
    
    # Step 1: Generate analogies in parallel
    print("\n--- STEP 1: GENERATION ---")
    gen_results = []
    
    tasks = [(m, mode) for m in models for mode in modes]
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(run_single_model_subprocess, m, mode): (m, mode)
            for m, mode in tasks
        }
        
        for future in tqdm(as_completed(futures), total=len(futures), desc="Generation"):
            result = future.result()
            gen_results.append(result)
    
    # Step 2: Evaluate in parallel
    print("\n--- STEP 2: EVALUATION ---")
    eval_results = []
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(evaluate_single_model_subprocess, m, mode): (m, mode)
            for m, mode in tasks
        }
        
        for future in tqdm(as_completed(futures), total=len(futures), desc="Evaluation"):
            result = future.result()
            eval_results.append(result)
    
    # Summary
    print("\n" + "="*70)
    print("PIPELINE EXECUTION SUMMARY")
    print("="*70)
    
    gen_df = pd.DataFrame(gen_results)
    eval_df = pd.DataFrame(eval_results)
    
    print("\nGeneration Results:")
    print(gen_df['status'].value_counts())
    
    print("\nEvaluation Results:")
    print(eval_df['status'].value_counts())
    
    return gen_df, eval_df

# Uncomment to run full pipeline
gen_results, eval_results = run_full_pipeline(max_workers=4)

FULL PIPELINE EXECUTION
Models: 12
Modes: ['targetonly', 'withsub']
Total runs: 24
Max workers: 4

--- STEP 1: GENERATION ---
Starting: gpt-oss-20b | targetonly
Starting: gpt-oss-20b | withsub
Starting: gpt-oss-120b | targetonly
Starting: gpt-oss-120b | withsub


Generation:   0%|          | 0/24 [00:00<?, ?it/s]

In [ ]:
# =============================================================================
# CELL 5: Aggregate All Results
# =============================================================================

def aggregate_all_results():
    """
    Aggregate evaluation results from all model runs.
    """
    print("="*70)
    print("AGGREGATING ALL RESULTS")
    print("="*70)
    
    # Find all evaluation files
    eval_files = glob.glob(os.path.join(RESULTS_DIR, "*_eval.csv"))
    print(f"Found {len(eval_files)} evaluation files")
    
    if len(eval_files) == 0:
        print("No evaluation files found. Run the pipeline first.")
        return None
    
    all_results = []
    
    for eval_file in eval_files:
        filename = os.path.basename(eval_file)
        
        # Parse model and mode from filename
        # Format: LLM_{model}_{mode}_eval.csv
        parts = filename.replace("_eval.csv", "").split("_")
        if len(parts) >= 3:
            mode = parts[-1]  # Last part is mode
            model = "_".join(parts[1:-1])  # Everything between LLM and mode
        else:
            continue
        
        try:
            df = pd.read_csv(eval_file)
            df['model'] = model
            df['mode'] = mode
            all_results.append(df)
            print(f"  Loaded: {filename} ({len(df)} records)")
        except Exception as e:
            print(f"  Error loading {filename}: {e}")
    
    if len(all_results) == 0:
        print("No valid results found.")
        return None
    
    # Combine all results
    combined_df = pd.concat(all_results, ignore_index=True)
    print(f"\nTotal aggregated records: {len(combined_df)}")
    
    # Save aggregated results
    output_path = os.path.join(RESULTS_DIR, "aggregated_results.csv")
    combined_df.to_csv(output_path, index=False)
    print(f"Saved aggregated results to: {output_path}")
    
    return combined_df


def compute_summary_metrics(df: pd.DataFrame):
    """
    Compute summary metrics for each model/mode combination.
    Updated to use new column structure: gold_ranks_list, sem_gold_ranks_list
    """
    if df is None or len(df) == 0:
        return None
    
    print("\n" + "="*70)
    print("COMPUTING SUMMARY METRICS")
    print("="*70)
    
    # Filter successful evaluations
    successful = df[df['status'] == 'success'].copy()
    print(f"Successful evaluations: {len(successful)} / {len(df)}")
    
    # Helper to get best rank from ranks list (JSON string)
    def get_best_rank(ranks_json: str) -> int:
        """Get best (lowest) rank from JSON list, or -1 if empty."""
        try:
            if pd.isna(ranks_json) or ranks_json == '':
                return -1
            ranks = json.loads(ranks_json)
            return min(ranks) if ranks else -1
        except:
            return -1
    
    # Compute best ranks for each row
    successful['best_exact_rank'] = successful['gold_ranks_list'].apply(get_best_rank)
    successful['best_sem_rank'] = successful['sem_gold_ranks_list'].apply(get_best_rank)
    
    # Effective rank: exact if found, else semantic
    def get_effective_rank(row):
        if row['best_exact_rank'] != -1:
            return row['best_exact_rank']
        return row['best_sem_rank']
    
    successful['effective_rank'] = successful.apply(get_effective_rank, axis=1)
    
    # Group by model and mode
    metrics = []
    
    for (model, mode), group in successful.groupby(['model', 'mode']):
        total = len(group)
        
        # LLM Judge scores
        coherence = group['analogy_coherence'].mean()
        mapping = group['mapping_soundness'].mean()
        explanatory = group['explanatory_power'].mean()
        avg_score = group['average_score'].mean()
        
        # Hit@K exact (using best rank from all gold sources)
        hit1_exact = (group['best_exact_rank'] == 1).sum() / total
        hit2_exact = (group['best_exact_rank'].between(1, 2)).sum() / total
        hit3_exact = (group['best_exact_rank'].between(1, 3)).sum() / total
        
        # Hit@K semantic (using effective rank: exact if found, else semantic)
        hit1_sem = (group['effective_rank'] == 1).sum() / total
        hit2_sem = (group['effective_rank'].between(1, 2)).sum() / total
        hit3_sem = (group['effective_rank'].between(1, 3)).sum() / total
        
        # Top-1 embedding metrics (if available)
        top1_embedding_avg_score = None
        if 'top1_embedding_score' in group.columns:
            top1_embedding_avg_score = group['top1_embedding_score'].mean()
        
        # Judge scores for top1_embedding (from judge_embedding column)
        embedding_judge_avg = None
        if 'judge_embedding' in group.columns:
            def get_judge_avg(judge_json):
                try:
                    if pd.isna(judge_json) or judge_json == '':
                        return None
                    judge = json.loads(judge_json)
                    return judge.get('average', None)
                except:
                    return None
            judge_avgs = group['judge_embedding'].apply(get_judge_avg).dropna()
            if len(judge_avgs) > 0:
                embedding_judge_avg = judge_avgs.mean()
        
        metrics.append({
            'model': model,
            'mode': mode,
            'n_records': total,
            # LLM Judge scores for top1_baseline
            'coherence': coherence,
            'mapping': mapping,
            'explanatory': explanatory,
            'avg_score': avg_score,  # Judge average for baseline
            # LLM Judge score for top1_embedding
            'embedding_judge_avg': embedding_judge_avg,  # Judge average for embedding selection
            # Hit metrics
            'hit@1_exact': hit1_exact,
            'hit@2_exact': hit2_exact,
            'hit@3_exact': hit3_exact,
            'hit@1_semantic': hit1_sem,
            'hit@2_semantic': hit2_sem,
            'hit@3_semantic': hit3_sem,
            # Embedding similarity score
            'top1_embedding_avg_score': top1_embedding_avg_score,  # Average embedding similarity score
        })
    
    metrics_df = pd.DataFrame(metrics)
    
    # Sort by avg_score descending
    metrics_df = metrics_df.sort_values('avg_score', ascending=False)
    
    # Save metrics
    output_path = os.path.join(RESULTS_DIR, "summary_metrics.csv")
    metrics_df.to_csv(output_path, index=False)
    print(f"Saved summary metrics to: {output_path}")
    
    return metrics_df


# Aggregate and compute metrics
aggregated_df = aggregate_all_results()
if aggregated_df is not None:
    summary_metrics = compute_summary_metrics(aggregated_df)
    if summary_metrics is not None:
        print("\nSummary Metrics:")
        display_cols = ['model', 'mode', 'avg_score', 'hit@1_exact', 'hit@1_semantic']
        if 'top1_embedding_avg_score' in summary_metrics.columns:
            display_cols.append('top1_embedding_avg_score')
        print(summary_metrics[display_cols].to_string(index=False))

In [ ]:
# =============================================================================
# CELL 6: Visualization - Model Comparison
# =============================================================================

# Thesis Color Palette: Coral Orange & Steel Blue (Sans-Serif)
COLORS = {
    'primary': '#4682B4',      # Steel Blue
    'secondary': '#FF7F50',    # Coral Orange
    'accent1': '#5A9BD4',      # Light Steel Blue
    'accent2': '#FF9966',      # Light Coral
    'dark': '#2F5496',         # Dark Steel Blue
    'text': '#333333',         # Dark Gray
    'light': '#B0C4DE',        # Light Steel Blue (muted)
}

PALETTE_MAIN = [COLORS['primary'], COLORS['secondary'], COLORS['accent1'], 
                COLORS['accent2'], COLORS['dark'], COLORS['light'],
                '#6495ED', '#FFA07A', '#87CEEB', '#FA8072', '#4169E1', '#CD5C5C']

# Set global matplotlib style - Sans-Serif for Thesis
plt.style.use('seaborn-v0_8-white')
rcParams['font.family'] = 'sans-serif'
rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans', 'Calibri']
rcParams['font.size'] = 14
rcParams['axes.labelsize'] = 16
rcParams['axes.titlesize'] = 18
rcParams['axes.titleweight'] = 'bold'
rcParams['figure.dpi'] = 150
rcParams['savefig.dpi'] = 300
rcParams['savefig.bbox'] = 'tight'
rcParams['axes.spines.top'] = False
rcParams['axes.spines.right'] = False


def plot_model_comparison(metrics_df: pd.DataFrame):
    """
    Create visualization comparing all models.
    """
    if metrics_df is None or len(metrics_df) == 0:
        print("No metrics to visualize")
        return
    
    # Create output directory for figures
    fig_dir = os.path.join(RESULTS_DIR, 'figures')
    os.makedirs(fig_dir, exist_ok=True)
    
    # =============================================================================
    # Figure 1: Average Score by Model and Mode
    # =============================================================================
    fig1, ax1 = plt.subplots(figsize=(14, 6))
    
    # Pivot for grouped bar chart
    pivot_df = metrics_df.pivot(index='model', columns='mode', values='avg_score')
    pivot_df = pivot_df.sort_values('targetonly', ascending=False)
    
    x = np.arange(len(pivot_df))
    width = 0.35
    
    bars1 = ax1.bar(x - width/2, pivot_df['targetonly'], width, 
                    label='Target Only', color=COLORS['primary'])
    bars2 = ax1.bar(x + width/2, pivot_df['withsub'], width,
                    label='With Sub-concepts', color=COLORS['secondary'])
    
    ax1.set_xlabel('Model')
    ax1.set_ylabel('Average LLM Judge Score (1-3)')
    ax1.set_title('LLM Baselines: Average Quality Score by Model')
    ax1.set_xticks(x)
    ax1.set_xticklabels(pivot_df.index, rotation=45, ha='right')
    ax1.legend()
    ax1.set_ylim(0, 3.2)
    ax1.axhline(y=2, color=COLORS['text'], linestyle='--', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, 'avg_score_comparison.png'))
    plt.savefig(os.path.join(fig_dir, 'avg_score_comparison.pdf'))
    plt.show()
    
    # =============================================================================
    # Figure 2: Hit@K Comparison (Exact vs Semantic)
    # =============================================================================
    fig2, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Exact Hit@K
    ax2 = axes[0]
    hit_cols = ['hit@1_exact', 'hit@2_exact', 'hit@3_exact']
    hit_labels = ['Hit@1', 'Hit@2', 'Hit@3']
    
    mode_df = metrics_df[metrics_df['mode'] == 'targetonly'].copy()
    mode_df = mode_df.sort_values('avg_score', ascending=False)
    
    x = np.arange(len(mode_df))
    width = 0.25
    
    for i, (col, label) in enumerate(zip(hit_cols, hit_labels)):
        ax2.bar(x + i*width, mode_df[col] * 100, width, 
               label=label, color=PALETTE_MAIN[i])
    
    ax2.set_xlabel('Model')
    ax2.set_ylabel('Hit Rate (%)')
    ax2.set_title('Hit@K (Exact Match) - Target Only Mode')
    ax2.set_xticks(x + width)
    ax2.set_xticklabels(mode_df['model'], rotation=45, ha='right')
    ax2.legend()
    
    # Semantic Hit@K
    ax3 = axes[1]
    hit_cols = ['hit@1_semantic', 'hit@2_semantic', 'hit@3_semantic']
    
    for i, (col, label) in enumerate(zip(hit_cols, hit_labels)):
        ax3.bar(x + i*width, mode_df[col] * 100, width,
               label=label, color=PALETTE_MAIN[i])
    
    ax3.set_xlabel('Model')
    ax3.set_ylabel('Hit Rate (%)')
    ax3.set_title('Hit@K (Semantic, threshold >= 0.7) - Target Only Mode')
    ax3.set_xticks(x + width)
    ax3.set_xticklabels(mode_df['model'], rotation=45, ha='right')
    ax3.legend()
    
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, 'hitk_comparison.png'))
    plt.savefig(os.path.join(fig_dir, 'hitk_comparison.pdf'))
    plt.show()
    
    # =============================================================================
    # Figure 3: Score Dimensions Breakdown
    # =============================================================================
    fig3, ax4 = plt.subplots(figsize=(14, 6))
    
    score_cols = ['coherence', 'mapping', 'explanatory']
    score_labels = ['Coherence', 'Mapping', 'Explanatory']
    
    x = np.arange(len(mode_df))
    width = 0.25
    
    for i, (col, label) in enumerate(zip(score_cols, score_labels)):
        ax4.bar(x + i*width, mode_df[col], width, label=label, color=PALETTE_MAIN[i])
    
    ax4.set_xlabel('Model')
    ax4.set_ylabel('Score (1-3)')
    ax4.set_title('LLM Judge Score Breakdown by Dimension (Target Only Mode)')
    ax4.set_xticks(x + width)
    ax4.set_xticklabels(mode_df['model'], rotation=45, ha='right')
    ax4.legend()
    ax4.set_ylim(0, 3.2)
    
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, 'score_dimensions.png'))
    plt.savefig(os.path.join(fig_dir, 'score_dimensions.pdf'))
    plt.show()
    
    # =============================================================================
    # Figure 4: Top-1 Embedding Similarity (Target-Analogy Comparison)
    # =============================================================================
    if 'top1_embedding_avg_score' in metrics_df.columns:
        fig4, ax5 = plt.subplots(figsize=(14, 6))
        
        # Pivot for grouped bar chart
        pivot_emb = metrics_df.pivot(index='model', columns='mode', values='top1_embedding_avg_score')
        pivot_emb = pivot_emb.sort_values('targetonly', ascending=False)
        
        x = np.arange(len(pivot_emb))
        width = 0.35
        
        bars1 = ax5.bar(x - width/2, pivot_emb['targetonly'], width, 
                        label='Target Only', color=COLORS['primary'])
        bars2 = ax5.bar(x + width/2, pivot_emb['withsub'], width,
                        label='With Sub-concepts', color=COLORS['secondary'])
        
        ax5.set_xlabel('Model')
        ax5.set_ylabel('Average Embedding Similarity Score')
        ax5.set_title('Top-1 Embedding Similarity: Target vs Generated Analogies')
        ax5.set_xticks(x)
        ax5.set_xticklabels(pivot_emb.index, rotation=45, ha='right')
        ax5.legend()
        ax5.set_ylim(0, 1.0)
        
        plt.tight_layout()
        plt.savefig(os.path.join(fig_dir, 'top1_embedding_comparison.png'))
        plt.savefig(os.path.join(fig_dir, 'top1_embedding_comparison.pdf'))
        plt.show()
    
    # =============================================================================
    # Figure 5: LLM Judge Score Comparison (Baseline vs Embedding Selection)
    # =============================================================================
    if 'embedding_judge_avg' in metrics_df.columns:
        fig5, ax6 = plt.subplots(figsize=(14, 6))
        
        # Prepare data: compare avg_score (baseline) vs embedding_judge_avg
        compare_df = metrics_df[metrics_df['mode'] == 'targetonly'].copy()
        compare_df = compare_df.sort_values('avg_score', ascending=False)
        
        x = np.arange(len(compare_df))
        width = 0.35
        
        bars1 = ax6.bar(x - width/2, compare_df['avg_score'], width, 
                        label='Baseline Selection (Model Order)', color=COLORS['primary'])
        bars2 = ax6.bar(x + width/2, compare_df['embedding_judge_avg'], width,
                        label='Embedding Selection (Similarity)', color=COLORS['secondary'])
        
        ax6.set_xlabel('Model')
        ax6.set_ylabel('Average LLM Judge Score (1-3)')
        ax6.set_title('LLM Judge: Baseline vs Embedding Selection (Target Only Mode)')
        ax6.set_xticks(x)
        ax6.set_xticklabels(compare_df['model'], rotation=45, ha='right')
        ax6.legend()
        ax6.set_ylim(0, 3.2)
        ax6.axhline(y=2, color=COLORS['text'], linestyle='--', alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(os.path.join(fig_dir, 'judge_baseline_vs_embedding.png'))
        plt.savefig(os.path.join(fig_dir, 'judge_baseline_vs_embedding.pdf'))
        plt.show()
    
    print(f"\nFigures saved to: {fig_dir}")


# Generate visualizations if metrics exist
if 'summary_metrics' in dir() and summary_metrics is not None:
    plot_model_comparison(summary_metrics)

In [ ]:
# =============================================================================
# CELL 7: Final Comparison Table
# =============================================================================

def create_final_comparison_table(metrics_df: pd.DataFrame):
    """
    Create a comprehensive comparison table for all models.
    """
    if metrics_df is None or len(metrics_df) == 0:
        print("No metrics available")
        return None
    
    print("="*100)
    print("FINAL MODEL COMPARISON TABLE")
    print("="*100)
    
    # Format for display
    display_df = metrics_df.copy()
    
    # Round numeric columns
    for col in ['coherence', 'mapping', 'explanatory', 'avg_score', 'embedding_judge_avg']:
        if col in display_df.columns:
            display_df[col] = display_df[col].round(2)
    
    for col in ['hit@1_exact', 'hit@2_exact', 'hit@3_exact', 
                'hit@1_semantic', 'hit@2_semantic', 'hit@3_semantic']:
        if col in display_df.columns:
            display_df[col] = (display_df[col] * 100).round(1).astype(str) + '%'
    
    # Round top1_embedding_avg_score if available
    if 'top1_embedding_avg_score' in display_df.columns:
        display_df['top1_embedding_avg_score'] = display_df['top1_embedding_avg_score'].round(4)
    
    # Sort by avg_score
    display_df = display_df.sort_values('avg_score', ascending=False)
    
    print("\n--- LLM Judge Quality Scores (Top-1 Baseline) ---")
    quality_cols = ['model', 'mode', 'n_records', 'coherence', 'mapping', 'explanatory', 'avg_score']
    print(display_df[quality_cols].to_string(index=False))
    
    # Show embedding judge scores if available
    if 'embedding_judge_avg' in display_df.columns:
        print("\n--- LLM Judge Quality Scores (Top-1 Embedding) ---")
        emb_judge_cols = ['model', 'mode', 'avg_score', 'embedding_judge_avg']
        print("  avg_score = baseline selection, embedding_judge_avg = embedding selection")
        print(display_df[emb_judge_cols].to_string(index=False))
    
    print("\n--- Hit@K Metrics ---")
    hit_cols = ['model', 'mode', 'hit@1_exact', 'hit@2_exact', 'hit@3_exact',
                'hit@1_semantic', 'hit@2_semantic', 'hit@3_semantic']
    print(display_df[hit_cols].to_string(index=False))
    
    # Show top1_embedding scores if available
    if 'top1_embedding_avg_score' in display_df.columns:
        print("\n--- Top-1 Embedding Scores (Target-Analogy Similarity) ---")
        emb_cols = ['model', 'mode', 'top1_embedding_avg_score']
        print(display_df[emb_cols].to_string(index=False))
    
    # Identify best models
    print("\n" + "="*100)
    print("KEY FINDINGS")
    print("="*100)
    
    best_quality = display_df.loc[display_df['avg_score'].idxmax()]
    print(f"\nBest LLM Quality Score: {best_quality['model']} ({best_quality['mode']})")
    print(f"  Average Score: {best_quality['avg_score']}")
    
    # Compare targetonly vs withsub
    print("\n--- Mode Comparison (averaged across models) ---")
    agg_cols = {
        'avg_score': 'mean',
        'hit@1_exact': 'mean',
        'hit@1_semantic': 'mean'
    }
    if 'embedding_judge_avg' in metrics_df.columns:
        agg_cols['embedding_judge_avg'] = 'mean'
    if 'top1_embedding_avg_score' in metrics_df.columns:
        agg_cols['top1_embedding_avg_score'] = 'mean'
    
    mode_comparison = metrics_df.groupby('mode').agg(agg_cols).round(3)
    print(mode_comparison)
    
    return display_df


# Create final comparison
if 'summary_metrics' in dir() and summary_metrics is not None:
    final_table = create_final_comparison_table(summary_metrics)